# 04 — SHAP Explainability

**This is the contribution over the source paper.** Periáñez et al. ranked
features by single-feature predictive power, which tells you *that* a feature
matters but not:

- which **direction** it pushes (does more spending protect retention or signal it?)
- how the effect **changes across the range** (is the first purchase what matters, or the tenth?)
- why an **individual player** was flagged — which is what a retention system must act on

SHAP answers all three by assigning each feature a signed contribution to every
individual prediction, with the guarantee that contributions sum to the
prediction itself.

In [1]:
import sys, warnings
from pathlib import Path

# Allow `from src import ...` when the notebook is run from notebooks/
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)
print("project root:", ROOT)

project root: D:\bhanu_fyp\player-churn-ml


In [2]:
import shap
from IPython.display import Image
from src import config, labeling, features, modeling, explainability

GAME = "game1"
OP, CP = config.OBSERVATION_PERIOD_DAYS, config.CHURN_PREDICTION_PERIOD_DAYS

events = pd.read_parquet(config.INTERIM_DIR / f"{GAME}_events.parquet")
cohort = labeling.build_cohort(events, OP, CP)
feats = features.prune_features(features.build_features(events, cohort, OP))
split = modeling.make_split(feats)

  cohort: 25,956 players -> 25,422 after left guard -> 25,121 after right guard -> 25,121 with OP activity | churn rate 92.6%


  dropped 2 uninformative columns: std_gap_hours, activity_slope


We explain the **XGBoost** model specifically. Even where another model edges it
on AUC, `TreeExplainer` computes exact Shapley values for tree ensembles in
polynomial time, so the explanations are exact rather than sampled.

In [3]:
model = modeling.build_models(split.y_train)["xgboost"]
model.fit(split.X_train, split.y_train)

explainer = shap.TreeExplainer(model)
sv = explainer(split.X_test, check_additivity=False)
print("SHAP values:", sv.values.shape)

SHAP values: (5025, 29)


## Global importance — which systems retain players?

The beeswarm shows every player as a dot. Position on the x-axis is that
feature's push toward churn (right) or retention (left); colour is the feature's
value. A red band on the left means *high values of this feature protect
retention*.

In [4]:
shap.plots.beeswarm(sv, max_display=18, show=False)
plt.title(f"{GAME} — SHAP summary (right = pushes toward churn)")
plt.tight_layout()
plt.show()

In [5]:
shap.plots.bar(sv, max_display=18, show=False)
plt.title(f"{GAME} — mean |SHAP|")
plt.tight_layout()
plt.show()

## Ranked importance with direction

In [6]:
importance = explainability.explain(model, split.X_test, GAME, "xgboost")
importance.head(20).round(4)

  SHAP top features: recency_hours(chu), active_duration_hours(ret), score_median(chu), score_best(ret), score_worst(ret), weekend_ratio(ret)


,feature,mean_abs_shap,direction,pushes_toward
0,recency_hours,0.3805,1.0,churn
1,active_duration_hours,0.3764,-1.0,retention
2,score_median,0.2050,1.0,churn
3,score_best,0.1815,-1.0,retention
4,score_worst,0.1749,-1.0,retention
5,weekend_ratio,0.1628,-1.0,retention
6,score_mean,0.1474,-1.0,retention
7,mean_gap_hours,0.1383,1.0,churn
8,total_playtime_minutes,0.0991,-1.0,retention
9,score_std,0.0975,-1.0,retention


## Dependence plots — the *shape* of each effect

Correlation gives one number per feature. These show the whole curve, including
thresholds and saturation points, which is where actionable design decisions
live (e.g. "risk spikes once the gap since last session passes X hours").

In [7]:
for feature in importance["feature"].head(4):
    shap.plots.scatter(sv[:, feature], show=False)
    plt.title(f"{GAME} — {feature}")
    plt.tight_layout()
    plt.show()

## Individual explanations

This is what makes a churn score operational. For a single player the waterfall
shows exactly which behaviours pushed them toward leaving — so an intervention
can target the actual cause rather than sending everyone the same reward.

In [8]:
totals = sv.values.sum(axis=1)
for label, pos in [("Highest-risk player", int(np.argmax(totals))),
                   ("Lowest-risk player", int(np.argmin(totals)))]:
    shap.plots.waterfall(sv[pos], max_display=14, show=False)
    plt.title(f"{GAME} — {label}")
    plt.tight_layout()
    plt.show()

### From explanation to intervention

Mapping the top SHAP drivers onto concrete retention levers is the
"engagement optimisation" half of the research question.

In [9]:
levers = {
    "recency_hours":          "Re-engagement push before the gap becomes terminal",
    "max_gap_hours":          "Streak / daily-login mechanics to prevent long gaps",
    "session_count":          "Session-start incentives; reduce friction to launch",
    "active_days":            "Multi-day reward ladders",
    "consecutive_play_ratio": "Habit-forming daily objectives",
    "score_mean":             "Difficulty tuning — keep scores in the flow channel",
    "score_improvement":      "Visible mastery feedback; surface personal bests",
    "score_slope":            "Progression pacing — avoid early plateaus",
    "max_loss_streak":        "Dynamic difficulty / matchmaking after losing streaks",
    "fail_ratio":             "Ease early failure spikes; add assists",
    "quit_ratio":             "Investigate mid-attempt abandonment causes",
    "win_ratio":              "Matchmaking balance",
    "purchase_count":         "First-purchase conversion offers",
    "purchase_total":         "Value reinforcement for spenders",
    "first_purchase_day":     "Bring the first purchase forward",
    "mean_session_minutes":   "Content pacing for the observed session length",
}

top = importance.head(12).copy()
top["retention_lever"] = top["feature"].map(levers).fillna("— review manually")
top[["feature", "mean_abs_shap", "pushes_toward", "retention_lever"]].round(4)

,feature,mean_abs_shap,pushes_toward,retention_lever
0,recency_hours,0.3805,churn,Re-engagement push before the gap becomes term...
1,active_duration_hours,0.3764,retention,— review manually
2,score_median,0.2050,churn,— review manually
3,score_best,0.1815,retention,— review manually
4,score_worst,0.1749,retention,— review manually
5,weekend_ratio,0.1628,retention,— review manually
6,score_mean,0.1474,retention,Difficulty tuning — keep scores in the flow ch...
7,mean_gap_hours,0.1383,churn,— review manually
8,total_playtime_minutes,0.0991,retention,— review manually
9,score_std,0.0975,retention,— review manually
